# Complaint Ingestion Pipeline — Phase 2 Day 1

**Goal:** Load `complaints.csv` → convert rows into LlamaIndex Documents → embed them → store in FAISS vector index → confirm retrieval works.

**What's happening under the hood (RAG Phase 1 = Indexing):**
```
CSV rows → Documents → Embedding Model → Vectors → FAISS Index
```

**Embedding model:** `all-MiniLM-L6-v2` (HuggingFace, free, local, 384-dim). No API key needed.

> Note: Groq doesn't offer embedding models (only chat LLMs). Groq will be used later for the ReAct agent's reasoning LLM. Embeddings stay on HuggingFace.

***Import the libraries***

In [24]:
import csv#built-in csv parser
from pathlib import Path#handle path

import faiss#facebook ai similarity search store vector and find nearest neighbor very fast => used here bcoz its industry standard
from llama_index.core import Document, Settings, VectorStoreIndex, StorageContext
#Document=> box that holds text+metadata . Only csv rows get wrapped in these
#Settings=> Global config. Set the embedder once everything uses it + saves you from passing it everywhere
#VectorStoreIndex: orchestrator=>take doc-> ask embedder to embed them-> store result in Faiss
#StorageContext: small config bundle saying "use this vector store for storage"
# seperation of concerns 
# Document hold data Settings has config. VectorStoreIndex orchestrate StorageContext routes storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
# class that wraps a HuggingFace Model and makes it feel like embed.model.get_text_embedding("some text") => free+ Local+ good enough
from llama_index.vector_stores.faiss import FaissVectorStore
#without it VectorStoreIndex would not know how to put vectors into FAISS

**Configure the embedding model**

In [15]:
embed_model=HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
#model is small to be run on our cpu and large enough to satisfy our usecase with lesser computation
#alternatives : openai textembedding: better quality but costs
            #bert-base-uncased: bigger slower and not better for our usecase
Settings.embed_model=embed_model
# we use settings because LlamaIndex has many classes that need to know how to embed text. With setting we configure 
#it once and LlamaIndex auto -picks this everytime
Settings.llm=None
#llamaIndex explicitly try to open the openAPI keys explicitly so we r saying to it thatwe r only doing embedding
#  today stop complaining abt missing LLMs
test_vector=embed_model.get_text_embedding("hydraulic press failure")
print(f"Embedding dimension: {len(test_vector)}")
#this is sanity check whether fail loudly and fal early

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LLM is explicitly disabled. Using MockLLM.
Embedding dimension: 384


**Load CSV into Documents**

In [16]:
CSV_PATH = Path("complaints.csv")
#text: what you want to search by
#metadata: what you returned with the result
# so only the complaints.txt is text rest all the three cols are kept as metadata
documents=[]
with open(CSV_PATH, "r") as f:# with tell to close the file once open even if there is an error
    reader=csv.DictReader(f)
    #reads the csv header and convert into dict not row[0], row[1], 
    #but like row["complaint_text"]
    for i, row in enumerate(reader):
        #enumerate gives us both the index and the row. we use i as a row_id in metadata
        doc=Document(
            text=row["complaint_text"],
            metadata={
                "product_area": row["product_area"],
                "root_cause_label": row["root_cause_label"],
                "capa_summary": row["capa_summary"],
                "row_id":i,
            },
        )
        documents.append(doc)
        #wra each row as a Document and collect them to a list. AT teh end documents is a list of 71 Documnet objects

**Build the FAISS index**

In [17]:
EMBED_DIM = 384
#number must match the embedding model ka output size as all-MINILM produce
#  384 dim vector 
faiss_index = faiss.IndexFlatL2(EMBED_DIM)
#so f u ever switch models changing this line keeps the code consistent
#Index=FAISS's word for "database of vectors"
#Flat=no compression no clustering . store every vector as is (brute force)
#L2= uses euclidean distance (tsraight line distance bw 2 points in space)=> default
#for 71 vectors brute force is good IndexIVF flat , INdexHNSW exist for million for vectors=> overkill

vector_store = FaissVectorStore(faiss_index=faiss_index)
#wrap the raw FAISS object in LlamaIndex's adapter. Now LlamaIndex can talk to it
storage_context = StorageContext.from_defaults(vector_store=vector_store)
#small object with slots for different storage backends:
    #vector store(where vector go)=> FAISS
    #Doc store(where Documents go)=> in-memory default
    # Index store(where index metadat goes)=> in-mem default
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    show_progress=True, # gives progress bar
)
# loop thru all 71 doc 
# for each : 
    # Grab doc.txt 
    # Call embed_model.get_text_embedding(doc.txt)=> get 384 numbers 
    # Store those numbers in FAISS 
    # Remember which metadata belong to which vector
# return an index object that knows how to query everything

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/71 [00:00<?, ?it/s]

**Test Retrieval**

In [20]:
retriever=index.as_retriever(similarity_top_k=5)
#one index can have many retriever each with different settings 
TEST_QUERY="HYDRAULIC press stopped working during production"
results= retriever.retrieve(TEST_QUERY)
# embed the query string => 384 numbers 
# --> Ask FAISS "find 5 stored vectors closest to this one" 
# --> Return matching Documents + their similarity scores
for i, node in enumerate(results):
    print(f"--- Result {i+1} | Score: {node.score:.4f} ---")
    print(f"Complaint: {node.text}")
    print(f"Root Cause: {node.metadata['root_cause_label']}")

#small value=> means very close whereas large means loosely related

--- Result 1 | Score: 0.8167 ---
Complaint: Hydraulic press #3 failed mid-cycle — lost pressure at 80% of required forming force. 15 parts deformed.
Root Cause: equipment_failure
--- Result 2 | Score: 1.0886 ---
Complaint: Air compressor delivering 85 PSI instead of required 100 PSI. Pneumatic actuators underperforming on press line.
Root Cause: equipment_failure
--- Result 3 | Score: 1.1741 ---
Complaint: PLC on Assembly Line 4 intermittently resetting — losing recipe parameters. Operators re-entering values manually.
Root Cause: equipment_failure
--- Result 4 | Score: 1.2335 ---
Complaint: CNC spindle bearing seized during production run. Tool crashed into workpiece. 3 parts destroyed.
Root Cause: equipment_failure
--- Result 5 | Score: 1.3052 ---
Complaint: Conveyor belt motor overheated and tripped breaker on Line 2. 45 minutes downtime, 12 WIP parts damaged from stoppage.
Root Cause: equipment_failure


**Second test query**: whether we can get some results for second query too

In [23]:
query2="parts coming out wrong dimensions from CNC machine"
results2=retriever.retrieve(query2)

# embed the query string => 384 numbers 
# --> Ask FAISS "find 5 stored vectors closest to this one" 
# --> Return matching Documents + their similarity scores
for i, node in enumerate(results2):
    print(f"--- Result {i+1} | Score: {node.score:.4f} ---")
    print(f"Complaint: {node.text}")
    print(f"Root Cause: {node.metadata['root_cause_label']}")

#small value=> means very close whereas large means loosely related

--- Result 1 | Score: 0.8932 ---
Complaint: Operator loaded incorrect program on CNC machine — Part #A vs Part #B. 12 parts machined wrong before detection.
Root Cause: human_error
--- Result 2 | Score: 1.0944 ---
Complaint: CNC lathe feed rate set to 120% override during night shift — not authorized per process sheet. Parts show tool marks.
Root Cause: process_deviation
--- Result 3 | Score: 1.1682 ---
Complaint: Solder paste stencil for Product X used on Product Y assembly. Wrong pad pattern — 100 boards affected.
Root Cause: human_error
--- Result 4 | Score: 1.1727 ---
Complaint: QC inspector signed off on first article inspection but measurement records show dimensions were not actually checked.
Root Cause: human_error
--- Result 5 | Score: 1.1745 ---
Complaint: PCB trace width insufficient for actual current load — trace heating observed during extended operation testing.
Root Cause: design_issue


complaints.csv
    ↓ (csv.DictReader)
rows as dicts
    ↓ (Document(...))
LlamaIndex Documents
    ↓ (VectorStoreIndex.from_documents + embed_model)
384-dim vectors + metadata
    ↓ (FaissVectorStore)
FAISS index (in RAM)
    ↓ (index.as_retriever)
retriever object
    ↓ (retriever.retrieve("query"))
top 5 matching Documents with scores
